# Solutions · 16 · Humidification and cooling towers

Worked solutions to the exercises of [notebook 16](../notebooks/16_humidification_and_cooling_towers.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engsep is missing): install it from GitHub
    import engsep
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engsep"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import integrate, optimize
from engsep import drying_evaporation as de
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

In [2]:
def ntu(T_hot, T_cold, T_air, RH, L_over_G):
    H_in = de.enthalpy_moist_air(T_air, RH * de.humidity_saturation(T_air)) / 1e3
    driving = lambda Tw: de.enthalpy_moist_air(Tw, de.humidity_saturation(Tw)) / 1e3 - (H_in + L_over_G * 4.186 * (Tw - T_cold))
    if min(driving(Tw) for Tw in np.linspace(T_cold, T_hot, 60)) <= 0:
        return np.inf                                                   # the operating line touches the saturation curve: a pinch
    return integrate.quad(lambda Tw: 4.186 / driving(Tw), T_cold, T_hot)[0]

## Exercise 1

Repeat the tower calculation for a humid summer day (34 degC, 70 % RH). What cold-water temperature can the same tower (same NTU) now reach, and by how much must the water rate drop (lower L/G) to hold 28 degC?

In [3]:
NTU_design = ntu(40.0, 28.0, 30.0, 0.4, 1.2)
T_wb = de.wet_bulb_temperature(34.0, 0.7 * de.humidity_saturation(34.0))
T_cold_new = optimize.brentq(lambda Tc: ntu(40.0, Tc, 34.0, 0.7, 1.2) - NTU_design, T_wb + 0.3, 39.0)
print(f"design NTU {NTU_design:.2f}; on a 34 degC / 70 % day (wet-bulb {T_wb:.1f} degC) the same tower delivers water at {T_cold_new:.1f} degC instead of 28")
LG_new = optimize.brentq(lambda lg: ntu(40.0, 28.0, 34.0, 0.7, lg) - NTU_design, 0.2, 1.2) if ntu(40.0, 28.0, 34.0, 0.7, 0.2) < NTU_design else None
print(f"to hold 28 degC the water rate must drop to L/G = {LG_new:.2f}" if LG_new else "28 degC cannot be held on that day at any water rate: the approach to wet-bulb would be too small")

design NTU 1.42; on a 34 degC / 70 % day (wet-bulb 29.3 degC) the same tower delivers water at 32.5 degC instead of 28
28 degC cannot be held on that day at any water rate: the approach to wet-bulb would be too small


On a humid summer day the wet-bulb temperature rises by six kelvin and the tower's cold water rises with it;
holding the design temperature would need the water rate cut - or, as the calculation may find, is not
possible at all because the approach would shrink below what any tower achieves. The design wet-bulb is
chosen from the hottest weeks for exactly this reason.

## Exercise 2

A counterflow tower has a packing characteristic KaV/L = 1.8 (L/G)^-0.6. Find the L/G at which it exactly meets the demand for cooling 40 -> 28 degC at 30 degC / 40 % RH, and the corresponding air flow per kg of water.

In [4]:
char = lambda lg: 1.8 * lg ** (-0.6)
LG = optimize.brentq(lambda lg: char(lg) - ntu(40.0, 28.0, 30.0, 0.4, lg), 0.3, 3.0)
print(f"the packing meets the demand at L/G = {LG:.2f}: {1/LG:.2f} kg of air per kg of water, KaV/L = {char(LG):.2f}")

the packing meets the demand at L/G = 1.32: 0.76 kg of air per kg of water, KaV/L = 1.53


The demand curve (Merkel's integral) rises with L/G and the packing's characteristic falls; their crossing
is the operating point, and the air rate follows. Tower selection is this one intersection.

## Exercise 3

**Implement it yourself:** evaluate Merkel's integral with Simpson's rule on ten intervals and compare with quad.

In [5]:
H_in = de.enthalpy_moist_air(30.0, 0.4 * de.humidity_saturation(30.0)) / 1e3
f = lambda Tw: 4.186 / (de.enthalpy_moist_air(Tw, de.humidity_saturation(Tw)) / 1e3 - (H_in + 1.2 * 4.186 * (Tw - 28.0)))
Tw = np.linspace(28.0, 40.0, 11)
h = Tw[1] - Tw[0]
simpson = h / 3 * (f(Tw[0]) + 4 * sum(f(t) for t in Tw[1:-1:2]) + 2 * sum(f(t) for t in Tw[2:-1:2]) + f(Tw[-1]))
print(f"Simpson (10 intervals): {simpson:.4f}; quad: {ntu(40.0, 28.0, 30.0, 0.4, 1.2):.4f}")

Simpson (10 intervals): 1.4151; quad: 1.4151


Simpson's rule on ten intervals matches adaptive quadrature to four decimals: the integrand is smooth, and
this is the method the classic cooling-tower charts were computed with.